# ImputeMissingValues Example

Impute missing values using configurable, data type–specific strategies.

The imputation pipeline fills missing values without modifying existing, non-missing data. Each column is imputed according to its detected data type and the strategy configured in `df.dg.config.missing`.

Unlike `clean()`, which performs deterministic transformations, or `optimize()`, which changes column data types, `impute()` estimates or substitutes missing values. The appropriate strategy depends on the dataset and the requirements of your analysis.

After imputation, a report describing the applied strategies and the number of values filled is available through `df.dg.report["impute"]`.

## Supported strategies

Each data type can be configured independently.

| Data type | Supported strategies                 |
| --------- | ------------------------------------ |
| Numeric   | `mean`, `median`, `mode`, `constant` |
| Boolean   | `mode`, `constant`                   |
| Category  | `mode`, `constant`                   |
| Datetime  | `ffill`, `bfill`, `constant`         |
| Text      | `mode`, `constant`, `empty`          |

## Configuration

The following options control the behavior of the imputation plugin:

| Option              | Description                                                   |
| ------------------- | ------------------------------------------------------------- |
| `enabled`           | Enable or disable the plugin.                                 |
| `numeric_strategy`  | Strategy used for numeric columns.                            |
| `numeric_constant`  | Constant value used with the `constant` numeric strategy.     |
| `boolean_strategy`  | Strategy used for boolean columns.                            |
| `boolean_constant`  | Constant value used with the `constant` boolean strategy.     |
| `category_strategy` | Strategy used for categorical columns.                        |
| `category_constant` | Constant value used with the `constant` categorical strategy. |
| `datetime_strategy` | Strategy used for datetime columns.                           |
| `datetime_constant` | Constant value used with the `constant` datetime strategy.    |
| `text_strategy`     | Strategy used for text columns.                               |
| `text_constant`     | Constant value used with the `constant` text strategy.        |

## Report

After calling `df.dg.impute()`, the plugin stores a report in:

```python
df.dg.report["impute"]
```

For each imputed column, the report includes:

* the strategy that was applied
* the number of missing values that were filled

This makes it easy to audit the changes performed during imputation.

## Notes

* Only missing values are modified; existing values are never changed.
* Each column is imputed independently based on its detected data type.
* Different strategies can be configured for different data types.
* The plugin performs deterministic operations for a given configuration and dataset.
* Imputation should generally be performed after cleaning and type optimization so that missing values and column types have already been standardized.


In [1]:
import pandas as pd
# notice the need to danda import to have accessor df
import danda # noqa: F401  # registers the pandas accessor
from danda.report_renderer import ReportRenderer

renderer = ReportRenderer()

In [2]:
df = pd.DataFrame({
    "EmployeeID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Age": [25, None, 35, 40, None, 29, 31, None],
    "IsActive": pd.Series(
        [True, None, True, False, True, None, True, False],
        dtype="boolean"
    ),
    "Department": pd.Series(
        ["HR", None, "IT", "HR", "Finance", None, "IT", "HR"],
        dtype="category"
    ),
    "JoinDate": pd.to_datetime([
        "2024-01-01",
        None,
        "2024-01-03",
        "2024-01-04",
        None,
        "2024-01-06",
        None,
        "2024-01-08"
    ]),
    "City": [
        "London",
        None,
        "Paris",
        "London",
        None,
        "Berlin",
        "London",
        None,
    ],
})

df

,EmployeeID,Age,IsActive,Department,JoinDate,City
0,1001,25.0,True,HR,2024-01-01,London
1,1002,NaN,<NA>,NaN,NaT,NaN
2,1003,35.0,True,IT,2024-01-03,Paris
3,1004,40.0,False,HR,2024-01-04,London
4,1005,NaN,True,Finance,NaT,NaN
5,1006,29.0,<NA>,NaN,2024-01-06,Berlin
6,1007,31.0,True,IT,NaT,London
7,1008,NaN,False,HR,2024-01-08,NaN


## Missing values analysis
Before we start let us do some analysis in the data frame to identify missing values.

From the report we could identify:
- Age: 3 (37.5%)
- City: 3 (37.5%)
- JoinDate: 3 (37.5%)
- Department: 2 (25.0%)
- IsActive: 2 (25.0%)

In [3]:
report = df.dg.analyze().dg.report
print(renderer.render(report))

Danda Report

Analyze
-------

ColumnSummaryPlugin
    Column Summary:
    
    EmployeeID
    Type: int64
    Missing: 0 (0%)
    Unique: 8
    
    Age
    Type: float64
    Missing: 3 (37%)
    Unique: 5
    
    IsActive
    Type: boolean
    Missing: 2 (25%)
    Unique: 2
    
    Department
    Type: category
    Missing: 2 (25%)
    Unique: 3
    
    JoinDate
    Type: datetime64[us]
    Missing: 3 (37%)
    Unique: 5
    
    City
    Type: str
    Missing: 3 (37%)
    Unique: 3

MissingValuesSummaryPlugin
    Missing Value Summary
    Rows: 8
    Columns: 6
    
    Missing cells: 13 (27.1%)
    Columns with missing: 5
    Rows with missing: 5
    Complete rows: 3 (37.5%)

MissingValuesReportPlugin
    Missing values detected:
    - Age: 3 (37.5%)
    - City: 3 (37.5%)
    - JoinDate: 3 (37.5%)
    - Department: 2 (25.0%)
    - IsActive: 2 (25.0%)

SparseRowsReportPlugin
    Rows with many missing values:
    - Row 1: 5/6 missing (83.3%)
    - Row 4: 3/6 missing (50.0%)

Exec

## Enable plugin

The **Impute Missing Values** plugin is disabled by default. To use it, enable the plugin in the configuration before calling `df.dg.impute()`.


In [4]:
df.dg.config.imputation.enabled = True

## Configure imputation strategies

In this example, the imputation strategy is configured separately for each data type:

* **Age** uses the **median** (31).
* **IsActive** uses the **mode** (True).
* **Department** uses the **mode** (HR).
* **JoinDate** uses **forward fill** (`ffill`).
* **City** uses the constant value **`"Unknown"`**.


In [5]:
# Configure the imputation strategies
df.dg.config.imputation.numeric_strategy = "median"
df.dg.config.imputation.boolean_strategy = "mode"
df.dg.config.imputation.category_strategy = "mode"
df.dg.config.imputation.datetime_strategy = "ffill"
df.dg.config.imputation.text_strategy = "constant"
df.dg.config.imputation.text_constant = "Unknown"

new_df = df.dg.impute()

new_df

,EmployeeID,Age,IsActive,Department,JoinDate,City
0,1001,25.0,True,HR,2024-01-01,London
1,1002,31.0,True,HR,2024-01-01,Unknown
2,1003,35.0,True,IT,2024-01-03,Paris
3,1004,40.0,False,HR,2024-01-04,London
4,1005,31.0,True,Finance,2024-01-04,Unknown
5,1006,29.0,True,HR,2024-01-06,Berlin
6,1007,31.0,True,IT,2024-01-06,London
7,1008,31.0,False,HR,2024-01-08,Unknown


## Report
This report records, for each column, the imputation strategy that was applied and the number of missing values that were filled.

In [6]:
# This will show what happen
print(renderer.render(new_df.dg.report["impute"]))

Danda Report

Imputation
----------

Filled missing values:
- Age: median (3) (37%)
- IsActive: mode (2) (25%)
- Department: mode (2) (25%)
- JoinDate: ffill (3) (37%)
- City: constant (3) (37%)


Chain
-----

plugins_count
    1

plugin_names
    ['ImputeMissingValuesPlugin']

memory_usage
before_bytes
    586

after_bytes
    606

saved_bytes
    -20

saved_percent
    -3.4129692832764507
